In [2]:
from project_utils import create_project_structure

create_project_structure("digital_humanities_war")

Created default config.yaml at: /Users/kylejonespatricia/time_series/digital_humanities_war/config.yaml


{'base': '/Users/kylejonespatricia/time_series/digital_humanities_war',
 'raw_data': '/Users/kylejonespatricia/time_series/digital_humanities_war/data/raw',
 'processed_data': '/Users/kylejonespatricia/time_series/digital_humanities_war/data/processed',
 'images': '/Users/kylejonespatricia/time_series/digital_humanities_war/images',
 'config': '/Users/kylejonespatricia/time_series/digital_humanities_war/config.yaml',
 'requirements': '/Users/kylejonespatricia/time_series/digital_humanities_war/requirements.txt'}

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

df = pd.read_csv("/Users/kylejonespatricia/Downloads/Transformed_Dataset_for_Regression.csv")

In [6]:
df.head()

,Unnamed: 0,year,negative,positive,Topic,decade,num_conflicts,avg_hostility,total_fatalities,life_expectancy,...,avg_hostility_lag1,total_fatalities_lag1,num_conflicts_lag5,avg_hostility_lag5,total_fatalities_lag5,num_conflicts_lag10,avg_hostility_lag10,total_fatalities_lag10,log_gdp_percap,log_sentiment
0,0,1918,0.05780,0.09935,Economy,1910,2.0,4.000000,0.0,53.90,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.678917,0.408344
1,1,1919,0.05210,0.09640,Economy,1910,1.0,4.000000,0.0,54.00,...,4.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,8.698316,0.670047
2,2,1920,0.04665,0.10955,Economy,1920,0.0,0.000000,0.0,54.10,...,4.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,8.717346,0.691884
3,3,1921,0.04510,0.09570,Economy,1920,3.0,2.666667,0.0,54.66,...,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,8.736020,0.638662
4,4,1922,0.04865,0.08690,Economy,1920,0.0,0.000000,0.0,55.22,...,2.666667,0.0,NaN,NaN,NaN,NaN,NaN,NaN,8.754353,0.639862


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load the dataset
data_file = "/Users/kylejonespatricia/Downloads/Transformed_Dataset_for_Regression.csv"
df = pd.read_csv(data_file)

# Convert 'year' column to datetime and sort
df["year"] = pd.to_datetime(df["year"], format="%Y")
df.sort_values("year", inplace=True)

# Handle missing values
df.fillna(0, inplace=True)

# Normalize term frequency data
term_freq_cols = [col for col in df.columns if "frequency" in col]
df[term_freq_cols] = (df[term_freq_cols] - df[term_freq_cols].min()) / (df[term_freq_cols].max() - df[term_freq_cols].min())

# Convert war-related categorical variables
df["total_fatalities"] = df["total_fatalities"].astype("category")

# Create lagged variables for war indicators
for lag in [1, 5, 10]:
    df[f"num_conflicts_lag{lag}"] = df["num_conflicts"].shift(lag)
    df[f"avg_hostility_lag{lag}"] = df["avg_hostility"].shift(lag)
    df[f"total_fatalities_lag{lag}"] = df["total_fatalities"].shift(lag)

# Drop NaN values from lagging
df.dropna(inplace=True)

# Run regression analysis
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic["log_sentiment"]
    X = pd.get_dummies(df_topic[["log_gdp_percap", "life_expectancy", topic.lower(), "num_conflicts", "avg_hostility", "total_fatalities"]], drop_first=True)
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    print(f"Regression Results for {topic}")
    print(model.summary())

# Run ARIMA time series analysis
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic["log_sentiment"]
    if len(y) > 20:
        model = ARIMA(y, order=(1, 0, 0)).fit()
        print(f"ARIMA Model Results for {topic}")
        print(model.summary())

# Run Machine Learning models
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic["log_sentiment"]
    X = df_topic[["log_gdp_percap", "life_expectancy", topic.lower() + "_frequency", "num_conflicts", "avg_hostility", "total_fatalities"]].fillna(0)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    print(f"Random Forest RMSE for {topic}: {mean_squared_error(y_test, y_pred_rf, squared=False)}")
    gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    gb.fit(X_train, y_train)
    y_pred_gb = gb.predict(X_test)
    print(f"Gradient Boosting RMSE for {topic}: {mean_squared_error(y_test, y_pred_gb, squared=False)}")


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

In [16]:
df.to_csv("sentiment.csv")


In [ ]:

# Handle missing values
df.fillna(0, inplace=True)

# Log transformation of skewed variables
df["log_gdp_percap"] = np.log1p(df["gdp_percap"])
df["log_sentiment"] = np.log1p(df["sentiment"])

# Normalize term frequency data
term_freq_cols = [col for col in df.columns if "frequency" in col]
df[term_freq_cols] = (df[term_freq_cols] - df[term_freq_cols].min()) / (df[term_freq_cols].max() - df[term_freq_cols].min())

# Convert war-related categorical variables
df["num_conflicts"] = df["num_conflicts"].astype("category")

# Create lagged variables for war indicators
for lag in [1, 5, 10]:
    df[f"num_conflicts_lag{lag}"] = df["num_conflicts"].shift(lag)
    df[f"avg_hostility_lag{lag}"] = df["avg_hostility"].shift(lag)
    df[f"total_fatalities_lag{lag}"] = df["total_fatalities"].shift(lag)

# Drop NaN values from lagging
df.dropna(inplace=True)

# Run regression analysis
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic["log_sentiment"]
    X = pd.get_dummies(df_topic[["log_gdp_percap", "life_expectancy", topic.lower() + "_frequency", "num_conflicts", "avg_hostility", "total_fatalities"]], drop_first=True)
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    print(f"Regression Results for {topic}")
    print(model.summary())

# Run ARIMA time series analysis
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic["log_sentiment"]
    if len(y) > 20:
        model = ARIMA(y, order=(1, 0, 0)).fit()
        print(f"ARIMA Model Results for {topic}")
        print(model.summary())

# Run Machine Learning models
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic["log_sentiment"]
    X = df_topic[["log_gdp_percap", "life_expectancy", topic.lower() + "_frequency", "num_conflicts", "avg_hostility", "total_fatalities"]].fillna(0)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    print(f"Random Forest RMSE for {topic}: {mean_squared_error(y_test, y_pred_rf, squared=False)}")
    gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    gb.fit(X_train, y_train)
    y_pred_gb = gb.predict(X_test)
    print(f"Gradient Boosting RMSE for {topic}: {mean_squared_error(y_test, y_pred_gb, squared=False)}")


In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load the dataset
data_file = "/Users/kylejonespatricia/Documents/sentiment.csv"
df = pd.read_csv(data_file, parse_dates=["Date"])

# Set 'Date' as the index and sort
df.set_index("Date", inplace=True)
df.sort_index(inplace=True)

# Handle missing values
df.fillna(0, inplace=True)

# Convert war-related categorical variables
df["total_fatalities"] = df["total_fatalities"].astype("category")

# Create lagged variables for war indicators
for lag in [1, 5, 10]:
    df[f"num_conflicts_lag{lag}"] = df["num_conflicts"].shift(lag)
    df[f"avg_hostility_lag{lag}"] = df["avg_hostility"].shift(lag)

# Drop NaN values from lagging
df.dropna(inplace=True)

dummies = pd.get_dummies(X, columns=["total_fatalities"], drop_first=True)

# Run regression analysis
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic["log_sentiment"]
    X = df_topic[["log_gdp_percap", "life_expectancy", topic.lower() + "_frequency", "num_conflicts", "avg_hostility", "total_fatalities"]], dummies
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    print(f"Regression Results for {topic}")
    print(model.summary())

# Run ARIMA time series analysis
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic["log_sentiment"]
    if len(y) > 20:
        model = ARIMA(y, order=(1, 0, 0)).fit()
        print(f"ARIMA Model Results for {topic}")
        print(model.summary())

# Run Machine Learning models
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic["log_sentiment"]
    X = df_topic[["log_gdp_percap", "life_expectancy", topic.lower() + "_frequency", "num_conflicts", "avg_hostility"]].fillna(0)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    print(f"Random Forest RMSE for {topic}: {mean_squared_error(y_test, y_pred_rf, squared=False)}")
    gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    gb.fit(X_train, y_train)
    y_pred_gb = gb.predict(X_test)
    print(f"Gradient Boosting RMSE for {topic}: {mean_squared_error(y_test, y_pred_gb, squared=False)}")


KeyError: "None of [Index(['total_fatalities'], dtype='object')] are in the [columns]"

In [26]:
df.head()

,Unnamed: 0,Topic,decade,num_conflicts,avg_hostility,total_fatalities,life_expectancy,democracy_frequency,economy_frequency,equality_frequency,...,log_sentiment,num_conflicts_lag1,avg_hostility_lag1,total_fatalities_lag1,num_conflicts_lag5,avg_hostility_lag5,total_fatalities_lag5,num_conflicts_lag10,avg_hostility_lag10,total_fatalities_lag10
Date,,,,,,,,,,,,,,,,,,,,,
1853-01-01,344,Democracy,1850,1.0,3.0,0.0,39.35,0.052376,0.051987,0.437101,...,0.642125,1.0,3.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0
1853-01-01,267,Liberty,1850,1.0,3.0,0.0,39.35,0.052376,0.051987,0.437101,...,0.474350,1.0,3.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0
1854-01-01,268,Liberty,1850,2.0,3.0,0.0,39.70,0.042536,0.045495,0.416885,...,0.691323,1.0,3.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0
1854-01-01,154,Freedom,1850,2.0,3.0,0.0,39.70,0.042536,0.045495,0.416885,...,0.641988,2.0,3.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0
1854-01-01,345,Democracy,1850,2.0,3.0,0.0,39.70,0.042536,0.045495,0.416885,...,0.692682,2.0,3.0,0.0,1.0,3.0,0.0,1.0,3.0,0.0


In [106]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load the dataset
data_file = "/Users/kylejonespatricia/Documents/sentiment.csv"
df = pd.read_csv(data_file, parse_dates=["Date"])

# Set 'Date' as the index and sort
df.set_index("Date", inplace=True)
df.sort_index(inplace=True)

# Convert war-related categorical variables
df["Topic"] = df["Topic"].astype("category")
df["total_fatalities"] = df["total_fatalities"].astype("category")
df["num_conflicts"] = pd.to_numeric(df["num_conflicts"], errors='coerce')
df["avg_hostility"] = pd.to_numeric(df["avg_hostility"], errors='coerce')

# Create dummy variables for total_fatalities and drop category for 0 fatalities
df = pd.get_dummies(df, columns=["total_fatalities"], drop_first=True, dtype='int')

# Create lagged variables for war indicators
for lag in [1, 5, 10]:
    df[f"num_conflicts_lag{lag}"] = df["num_conflicts"].shift(lag)
    df[f"avg_hostility_lag{lag}"] = df["avg_hostility"].shift(lag)

# Run regression analysis
for topic in df["Topic"].unique():
    df_topic = df[df["Topic"] == topic]
    y = df_topic['log_sentiment'].astype(float)  # Ensure y is defined
    X = df_topic[[ 'total_fatalities_1.0', 'total_fatalities_2.0', 'total_fatalities_6.0',
       'total_fatalities_9.0']]
    X = sm.add_constant(X)  # Add intercept

    model = sm.OLS(y, X).fit()
    print(f"Regression Results for {topic}")
    print(model.summary())


Regression Results for Democracy
                            OLS Regression Results                            
Dep. Variable:          log_sentiment   R-squared:                       0.031
Model:                            OLS   Adj. R-squared:                 -0.005
Method:                 Least Squares   F-statistic:                    0.8736
Date:                Sun, 16 Feb 2025   Prob (F-statistic):              0.482
Time:                        21:32:13   Log-Likelihood:                 101.19
No. Observations:                 113   AIC:                            -192.4
Df Residuals:                     108   BIC:                            -178.8
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------

In [98]:
df.columns

Index(['Unnamed: 0', 'Topic', 'decade', 'num_conflicts', 'avg_hostility',
       'life_expectancy', 'democracy_frequency', 'economy_frequency',
       'equality_frequency', 'freedom_frequency', 'justice_frequency',
       'liberty_frequency', 'log_gdp_percap', 'log_sentiment',
       'total_fatalities_1.0', 'total_fatalities_2.0', 'total_fatalities_6.0',
       'total_fatalities_9.0', 'num_conflicts_lag1', 'avg_hostility_lag1',
       'num_conflicts_lag5', 'avg_hostility_lag5', 'num_conflicts_lag10',
       'avg_hostility_lag10'],
      dtype='object')

In [78]:
df.head(30)

,Unnamed: 0,Topic,decade,num_conflicts,avg_hostility,life_expectancy,democracy_frequency,economy_frequency,equality_frequency,freedom_frequency,...,total_fatalities_1.0,total_fatalities_2.0,total_fatalities_6.0,total_fatalities_9.0,num_conflicts_lag1,avg_hostility_lag1,num_conflicts_lag5,avg_hostility_lag5,num_conflicts_lag10,avg_hostility_lag10
Date,,,,,,,,,,,,,,,,,,,,,
1851-01-01,342,Democracy,1850,0.0,0.0,38.65,0.070803,0.060103,0.468267,0.613080,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN
1851-01-01,64,Economy,1850,0.0,0.0,38.65,0.070803,0.060103,0.468267,0.613080,...,False,False,False,False,0.0,0.0,NaN,NaN,NaN,NaN
1851-01-01,151,Freedom,1850,0.0,0.0,38.65,0.070803,0.060103,0.468267,0.613080,...,False,False,False,False,0.0,0.0,NaN,NaN,NaN,NaN
1851-01-01,265,Liberty,1850,0.0,0.0,38.65,0.070803,0.060103,0.468267,0.613080,...,False,False,False,False,0.0,0.0,NaN,NaN,NaN,NaN
1852-01-01,266,Liberty,1850,1.0,3.0,39.00,0.063372,0.056304,0.434312,0.599111,...,False,False,False,False,0.0,0.0,NaN,NaN,NaN,NaN
1852-01-01,343,Democracy,1850,1.0,3.0,39.00,0.063372,0.056304,0.434312,0.599111,...,False,False,False,False,1.0,3.0,0.0,0.0,NaN,NaN
1852-01-01,152,Freedom,1850,1.0,3.0,39.00,0.063372,0.056304,0.434312,0.599111,...,False,False,False,False,1.0,3.0,0.0,0.0,NaN,NaN
1852-01-01,65,Economy,1850,1.0,3.0,39.00,0.063372,0.056304,0.434312,0.599111,...,False,False,False,False,1.0,3.0,0.0,0.0,NaN,NaN
1853-01-01,153,Freedom,1850,1.0,3.0,39.35,0.052376,0.051987,0.437101,0.619217,...,False,False,False,False,1.0,3.0,0.0,0.0,NaN,NaN
